# COS424 Milestone 2 - Data Wrangling

## Cleaning the initial csv

### Configuration and Setup

In [23]:
# Imports
from pathlib import Path
import unicodedata
import numpy as np
import pandas as pd

In [24]:
# Locate the repository from its root, notebooks/, or a deeper directory.
REPO_ROOT = next(
    (path for path in (Path.cwd(), *Path.cwd().parents)
     if (path / "data.csv").is_file() and (path / "notebooks").is_dir()),
    None,
)
if REPO_ROOT is None:
    raise FileNotFoundError("Open the notebook with this repository as the working directory.")

INPUT_PATH = REPO_ROOT / "data.csv"
OUTPUT_DIR = REPO_ROOT / "outputs"
OUTPUT_PATH = OUTPUT_DIR / "data_clean.csv"

# Save any additional generated files under this directory.
CLEANING_OUTPUT_DIR = OUTPUT_DIR / "data_cleaning"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
CLEANING_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ---- Settings ----------------------------------------------------------------
# Strings that really mean "missing" in text columns (compared case-insensitively)
MISSING_PLACEHOLDERS = ["none", "n/a", "na", "null", "nan", "-", "--", "?", "unknown", ""]

# Replace curly quotes/apostrophes with straight ones in names and titles.
# Makes joins with other sources (e.g. full review text) far more reliable.
NORMALIZE_QUOTES = True

# Drop rows where a key identifying field (artist or album) is missing
DROP_MISSING_KEY_FIELDS = True

# The 'key' column is an album average of a categorical value (0 = C, 1 = C#, ...),
# so it has no musical meaning. True drops it.
DROP_KEY_COLUMN = True

# What to do with rows whose audio features are identical to a different artist's
# album (likely a bad title-only match to Spotify):
#   "flag" -> keep values, add features_suspect = True
#   "nan"  -> flag AND blank the audio features
#   "drop" -> remove the rows entirely
SUSPECT_FEATURES_ACTION = "drop"

# Years between release and review after which a review is treated as a reissue/retrospective
REISSUE_GAP_YEARS = 2

# Fill missing genre / record label with "Unknown" (False keeps them as blank/NaN)
FILL_UNKNOWN_CATEGORIES = False

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 200)

# ---- Cleaning log ------------------------------------------------------------
cleaning_log = []

def log(step, detail, affected):
    """Record one cleaning action and print it."""
    cleaning_log.append({"step": step, "detail": detail, "affected": int(affected)})
    print(f"[{step}] {detail}: {int(affected):,}")

### Read and Verify the File

In [25]:
with open(INPUT_PATH, "rb") as f:
    raw_bytes = f.read()

# Encoding check: fails loudly if the file is not valid UTF-8
try:
    raw_text = raw_bytes.decode("utf-8")
    encoding = "utf-8"
except UnicodeDecodeError:
    raw_text = raw_bytes.decode("latin-1")
    encoding = "latin-1"
print("Encoding:", encoding)
print("Has BOM:", raw_text.startswith("\ufeff"))

line_count = raw_text.count("\n") + (0 if raw_text.endswith("\n") else 1)

alb_data_raw = pd.read_csv(INPUT_PATH, encoding=encoding)
print(f"Lines in file: {line_count:,}  |  Rows parsed: {len(alb_data_raw):,}  |  Columns: {alb_data_raw.shape[1]}")

if line_count - 1 == len(alb_data_raw):
    print("OK: one row per line, no embedded line breaks.")
else:
    print("WARNING: row count differs from line count. Inspect quoted fields with line breaks.")

alb_data = alb_data_raw.copy()
alb_data.insert(0, "source_row", np.arange(len(alb_data)))   # keeps a link back to the raw file
ROWS_AT_START = len(alb_data)

Encoding: utf-8
Has BOM: False
Lines in file: 18,404  |  Rows parsed: 18,403  |  Columns: 18
OK: one row per line, no embedded line breaks.


### Initial Description of Data

In [26]:
print("Shape:", alb_data.shape)
display(alb_data.head())

Shape: (18403, 19)


,source_row,artist,album,reviewauthor,score,releaseyear,reviewdate,recordlabel,genre,danceability,energy,key,loudness,speechiness,acousticness,instrumentalness,liveness,valence,tempo
0,0,Studio 1,Studio 1,Andy Battaglia,8.5,2009.0,February 18 2009,Studio,Electronic,0.511917,0.499667,5.250000,-5.626583,0.031983,0.724917,0.024493,0.165367,0.555083,101.395167
1,1,John Fahey,The Great Santa Barbara Oil Slick,Mark Richardson,8.2,2005.0,February 13 2005,Water,Folk/Country,0.369765,0.325412,4.470588,-19.153824,0.148624,0.647053,0.559133,0.527782,0.179465,107.622647
2,2,Reigning Sound,Too Much Guitar,Stephen M. Deusner,8.3,2004.0,August 19 2004,In the Red,Electronic,0.253943,0.912857,4.428571,-1.089500,0.055500,0.000253,0.751214,0.199071,0.552786,133.895500
3,3,The Red Thread,After the Last,Chris Dahlen,7.3,2003.0,July 17 2003,Badman,Rock,0.425400,0.433474,5.700000,-12.871000,0.028260,0.310325,0.224137,0.125150,0.451400,104.354200
4,4,Mac Miller,Swimming,Evan Rytlewski,7.5,2018.0,August 3 2018,Warner Bros.,Rap,0.624846,0.438154,4.153846,-9.456077,0.170246,0.652462,0.012819,0.121131,0.281138,122.121308


In [27]:
profile = pd.DataFrame({
    "dtype": alb_data.dtypes.astype(str),
    "missing": alb_data.isna().sum(),
    "missing_%": (alb_data.isna().mean() * 100).round(2),
    "unique": alb_data.nunique(),
})
profile

,dtype,missing,missing_%,unique
source_row,int64,0,0.00,18403
artist,object,1,0.01,9049
album,object,0,0.00,17714
reviewauthor,object,0,0.00,564
score,float64,0,0.00,100
releaseyear,float64,325,1.77,61
reviewdate,object,0,0.00,4957
recordlabel,object,26,0.14,3286
genre,object,1289,7.00,10
danceability,float64,0,0.00,12407


In [28]:
alb_data.describe().T

,count,mean,std,min,25%,50%,75%,max
source_row,18403.0,9201.000000,5312.632838,0.00000,4600.500000,9201.000000,13801.500000,18402.000
score,18403.0,7.033255,1.251850,0.00000,6.500000,7.300000,7.800000,10.000
releaseyear,18078.0,2009.315411,7.708710,1957.00000,2005.000000,2010.000000,2015.000000,2019.000
danceability,18403.0,0.510635,0.159833,-1.00000,0.405784,0.510417,0.622000,0.974
energy,18403.0,0.600806,0.204257,-1.00000,0.471787,0.624364,0.750586,0.999
key,18403.0,5.215836,1.619708,-1.00000,4.333333,5.230769,6.100000,11.000
loudness,18403.0,-9.318390,4.264038,-51.72875,-11.248350,-8.492333,-6.416397,4.078
speechiness,18403.0,0.091125,0.092428,-1.00000,0.040704,0.056808,0.100029,0.958
acousticness,18403.0,0.302895,0.263730,-1.00000,0.089842,0.229806,0.463416,0.996
instrumentalness,18403.0,0.278616,0.300740,-1.00000,0.010060,0.153433,0.506816,0.982


In [29]:
TEXT_COLS = ["artist", "album", "reviewauthor", "recordlabel", "genre"]
FEATURE_COLS = ["danceability", "energy", "key", "loudness", "speechiness",
                "acousticness", "instrumentalness", "liveness", "valence", "tempo"]
UNIT_INTERVAL_COLS = ["danceability", "energy", "speechiness", "acousticness",
                      "instrumentalness", "liveness", "valence"]   # should lie in [0, 1]

### Exact Duplicate Rows

In [30]:
data_cols = [c for c in alb_data.columns if c != "source_row"]
dup_mask = alb_data.duplicated(subset=data_cols, keep="first")
display(alb_data[alb_data.duplicated(subset=data_cols, keep=False)][["source_row", "artist", "album", "reviewauthor", "reviewdate"]])
alb_data = alb_data[~dup_mask].copy()
log("Duplicates", "Exact duplicate rows removed", dup_mask.sum())

,source_row,artist,album,reviewauthor,reviewdate
8012,8012,Fiona Apple,When the Pawn...,Judy Berman,March 24 2019
8013,8013,Fiona Apple,When the Pawn...,Judy Berman,March 24 2019
12874,12874,Stick Men with Ray Guns,Some People Deserve to Suffer,Andy Beta,March 5 2003
12875,12875,Stick Men with Ray Guns,Some People Deserve to Suffer,Andy Beta,March 5 2003


[Duplicates] Exact duplicate rows removed: 2


### Text Normalization

In [31]:
QUOTE_MAP = str.maketrans({"\u201c": '"', "\u201d": '"', "\u2018": "'", "\u2019": "'",
                           "\u00a0": " "})   # also replaces non-breaking spaces

def normalize_text(s):
    if pd.isna(s):
        return s
    s = unicodedata.normalize("NFC", str(s))
    if NORMALIZE_QUOTES:
        s = s.translate(QUOTE_MAP)
    return " ".join(s.split())    # trims and collapses all runs of whitespace

for col in TEXT_COLS:
    before_unique = alb_data[col].nunique()
    new = alb_data[col].map(normalize_text)
    changed = (new.fillna("<NA>") != alb_data[col].fillna("<NA>")).sum()
    alb_data[col] = new
    log("Text normalisation", f"{col}: values changed ({before_unique:,} -> {alb_data[col].nunique():,} unique)", changed)

[Text normalisation] artist: values changed (9,049 -> 9,047 unique): 140
[Text normalisation] album: values changed (17,714 -> 17,711 unique): 150
[Text normalisation] reviewauthor: values changed (564 -> 559 unique): 83
[Text normalisation] recordlabel: values changed (3,286 -> 3,283 unique): 41
[Text normalisation] genre: values changed (10 -> 10 unique): 0
[Text normalisation] genre: values changed (10 -> 10 unique): 0


### Disguised Missing Values

In [32]:
placeholders = {p.lower() for p in MISSING_PLACEHOLDERS}

for col in TEXT_COLS:
    mask = alb_data[col].notna() & alb_data[col].str.lower().isin(placeholders)
    if mask.any():
        print(f"{col}: placeholder values found -> {alb_data.loc[mask, col].value_counts().to_dict()}")
    alb_data.loc[mask, col] = np.nan
    log("Disguised missing", f"{col}: placeholder strings -> NaN", mask.sum())

[Disguised missing] artist: placeholder strings -> NaN: 0
[Disguised missing] album: placeholder strings -> NaN: 0
[Disguised missing] reviewauthor: placeholder strings -> NaN: 0
recordlabel: placeholder values found -> {'none': 607, 'Unknown': 7}
[Disguised missing] recordlabel: placeholder strings -> NaN: 614
genre: placeholder values found -> {'none': 11}
[Disguised missing] genre: placeholder strings -> NaN: 11


### Inconsistent Category Spellings

In [33]:
def unify_case_variants(series):
    s = series.dropna()
    counts = s.value_counts()
    canonical = (counts.rename_axis("value").reset_index()
                       .assign(key=lambda d: d["value"].str.casefold())
                       .sort_values(["key", "count"], ascending=[True, False])
                       .drop_duplicates("key")
                       .set_index("key")["value"])
    groups = s.groupby(s.str.casefold()).agg(lambda x: sorted(set(x)))
    variants = groups[groups.str.len() > 1]
    return series.map(lambda v: canonical[v.casefold()] if pd.notna(v) else v), variants

for col in ["artist", "reviewauthor", "recordlabel", "genre"]:
    new, variants = unify_case_variants(alb_data[col])
    changed = (new.fillna("<NA>") != alb_data[col].fillna("<NA>")).sum()
    if len(variants):
        print(f"\n{col}: {len(variants)} case-variant groups, e.g. {variants.head(5).tolist()}")
    alb_data[col] = new
    log("Case variants", f"{col}: values unified to most common spelling", changed)


artist: 3 case-variant groups, e.g. [['Billy Woods', 'billy woods'], ['The Comet Is Coming', 'The Comet is Coming'], ['WIVES', 'Wives']]
[Case variants] artist: values unified to most common spelling: 3
[Case variants] reviewauthor: values unified to most common spelling: 0

recordlabel: 2 case-variant groups, e.g. [['GOOD', 'Good'], ['ITALIANS DO IT BETTER', 'Italians Do It Better']]
[Case variants] recordlabel: values unified to most common spelling: 3
[Case variants] genre: values unified to most common spelling: 0


### Data Types

In [34]:
parsed = pd.to_datetime(alb_data["reviewdate"], format="%B %d %Y", errors="coerce")
unparsed = parsed.isna() & alb_data["reviewdate"].notna()
if unparsed.any():
    print("Dates that could not be parsed:", alb_data.loc[unparsed, "reviewdate"].head().tolist())
alb_data["reviewdate"] = parsed
log("Types", "reviewdate parsed to datetime (failures)", unparsed.sum())

non_integer_years = (alb_data["releaseyear"].dropna() % 1 != 0).sum()
assert non_integer_years == 0, "releaseyear has non-integer values"
alb_data["releaseyear"] = alb_data["releaseyear"].astype("Int64")
log("Types", "releaseyear converted to nullable integer (missing values)", alb_data["releaseyear"].isna().sum())

for col in ["genre", "recordlabel"]:
    if FILL_UNKNOWN_CATEGORIES:
        n = alb_data[col].isna().sum()
        alb_data[col] = alb_data[col].fillna("Unknown")
        log("Types", f"{col}: missing filled with 'Unknown'", n)

print(alb_data[["reviewdate", "releaseyear"]].dtypes)

[Types] reviewdate parsed to datetime (failures): 0
[Types] releaseyear converted to nullable integer (missing values): 325
reviewdate     datetime64[ns]
releaseyear             Int64
dtype: object


### Missing Key Fields

In [35]:
key_missing = alb_data["artist"].isna() | alb_data["album"].isna()
display(alb_data.loc[key_missing, ["source_row", "artist", "album", "reviewauthor", "reviewdate"]])
if DROP_MISSING_KEY_FIELDS:
    alb_data = alb_data[~key_missing].copy()
    log("Missing keys", "Rows dropped (missing artist or album)", key_missing.sum())
else:
    log("Missing keys", "Rows with missing artist or album (kept)", key_missing.sum())

,source_row,artist,album,reviewauthor,reviewdate
4720,4720,NaN,Cellar EP,Kevin Lozano,2016-02-29


[Missing keys] Rows dropped (missing artist or album): 1


### Audio features: sentinels, ranges and the `key` column

In [36]:
sentinel_any = (alb_data[FEATURE_COLS] == -1).any(axis=1)
sentinel_all = (alb_data[FEATURE_COLS] == -1).all(axis=1)
print(f"Rows with -1 in all features: {sentinel_all.sum()}, in only some: {(sentinel_any & ~sentinel_all).sum()}")
alb_data[FEATURE_COLS] = alb_data[FEATURE_COLS].mask(alb_data[FEATURE_COLS] == -1)
log("Audio features", "Rows with -1 sentinel values set to NaN", sentinel_any.sum())

# Out-of-range values: a negative album average usually means one track carried the -1
# sentinel and was averaged in, so the value is contaminated and can't be trusted.
oor = (alb_data[UNIT_INTERVAL_COLS] < 0) | (alb_data[UNIT_INTERVAL_COLS] > 1)
print("\nValues outside [0, 1] after sentinel removal:\n", oor.sum().to_string())
if oor.any().any():
    display(alb_data.loc[oor.any(axis=1), ["artist", "album"] + UNIT_INTERVAL_COLS])
alb_data[UNIT_INTERVAL_COLS] = alb_data[UNIT_INTERVAL_COLS].mask(oor)
log("Audio features", "Out-of-range [0, 1] feature values set to NaN", oor.sum().sum())

alb_data["loudness_positive"] = alb_data["loudness"] > 0
alb_data["tempo_unusual"] = (alb_data["tempo"] < 50) | (alb_data["tempo"] > 250)
log("Audio features", "Rows flagged with positive loudness", alb_data["loudness_positive"].sum())
log("Audio features", "Rows flagged with unusual tempo (<50 or >250 BPM)", alb_data["tempo_unusual"].sum())

if DROP_KEY_COLUMN:
    non_int = (alb_data["key"].dropna() % 1 != 0).sum()
    print(f"\n'key' is non-integer (averaged) in {non_int:,} rows -> dropping column")
    alb_data = alb_data.drop(columns="key")
    FEATURE_COLS = [c for c in FEATURE_COLS if c != "key"]
    log("Audio features", "'key' column dropped (rows with averaged values)", non_int)

Rows with -1 in all features: 8, in only some: 0
[Audio features] Rows with -1 sentinel values set to NaN: 8

Values outside [0, 1] after sentinel removal:
 danceability        0
energy              0
speechiness         1
acousticness        0
instrumentalness    1
liveness            0
valence             0


,artist,album,danceability,energy,speechiness,acousticness,instrumentalness,liveness,valence
12570,Rahim,Ideal Lives,0.518364,0.531909,-0.058236,0.012756,-0.085783,0.075236,0.462364


[Audio features] Out-of-range [0, 1] feature values set to NaN: 2
[Audio features] Rows flagged with positive loudness: 9
[Audio features] Rows flagged with unusual tempo (<50 or >250 BPM): 5

'key' is non-integer (averaged) in 15,089 rows -> dropping column
[Audio features] 'key' column dropped (rows with averaged values): 15,089


### Audio features: suspected wrong matches

Two different artists' albums should essentially never have identical values across every audio feature. When they do (e.g. Muse's *Drones* and Nico Muhly's *Drones*), the features were most likely joined to Spotify by album title alone, so at least one row carries another album's audio.

It isn't possible to tell from this file which row is the correct one, so every row in such a group is flagged. What happens next is controlled by `SUSPECT_FEATURES_ACTION`.

In [37]:
has_features = alb_data[FEATURE_COLS].notna().all(axis=1)
feat_key = alb_data.loc[has_features, FEATURE_COLS].round(6).astype(str).agg("|".join, axis=1)
artists_per_vector = alb_data.loc[has_features].groupby(feat_key)["artist"].transform("nunique")

alb_data["features_suspect"] = False
alb_data.loc[artists_per_vector.index, "features_suspect"] = artists_per_vector > 1

examples = (alb_data[alb_data["features_suspect"]]
            .assign(_k=feat_key)
            .sort_values(["_k", "artist"])
            [["artist", "album", "energy", "tempo"]]
            .head(10))
display(examples)
log("Suspect features", "Rows sharing identical features with a different artist", alb_data["features_suspect"].sum())

if SUSPECT_FEATURES_ACTION == "nan":
    alb_data.loc[alb_data["features_suspect"], FEATURE_COLS] = np.nan
    log("Suspect features", "Suspect rows: audio features set to NaN", alb_data["features_suspect"].sum())
elif SUSPECT_FEATURES_ACTION == "drop":
    n = alb_data["features_suspect"].sum()
    alb_data = alb_data[~alb_data["features_suspect"]].copy()
    log("Suspect features", "Suspect rows dropped", n)

,artist,album,energy,tempo
4397,Muse,Drones,0.027630,81.425000
4398,Nico Muhly,Drones,0.027630,81.425000
12635,Nosaj Thing,Drift,0.079600,153.779000
12636,The Men,Drift,0.079600,153.779000
1506,Kath Bloom,Thin Thin Line,0.111600,82.949818
1505,Trans Am,The Red Line,0.111600,82.949818
6464,Hepcat,Out of Nowhere,0.637000,200.085000
6463,Jimi Tenor,Out of Nowhere,0.637000,200.085000
7870,Carmen Villain,Sleeper,0.123675,115.411750
7871,Ty Segall,Sleeper,0.123675,115.411750


[Suspect features] Rows sharing identical features with a different artist: 1,589
[Suspect features] Suspect rows dropped: 1,589


### Cross-field logic: release year vs review date

Single-column checks can't catch contradictions between columns.

- A review long after release is usually a reissue or retrospective. Those scores behave differently, so they are flagged with `is_reissue`.

- A release year after the review year is impossible in principle. These are mostly late-December reviews of albums dated the following year, so they are flagged rather than changed.

In [38]:
review_year = alb_data["reviewdate"].dt.year
gap = (review_year - alb_data["releaseyear"]).astype("Int64")
alb_data["years_since_release"] = gap

alb_data["is_reissue"] = (gap > REISSUE_GAP_YEARS).fillna(False).astype(bool)
alb_data["release_after_review"] = (gap < 0).fillna(False).astype(bool)

log("Cross-field", f"Reviews flagged as reissue (> {REISSUE_GAP_YEARS} years after release)", alb_data["is_reissue"].sum())
log("Cross-field", "Rows where release year is after review year (flagged)", alb_data["release_after_review"].sum())
display(alb_data.loc[alb_data["release_after_review"], ["artist", "album", "releaseyear", "reviewdate"]].head())

[Cross-field] Reviews flagged as reissue (> 2 years after release): 786
[Cross-field] Rows where release year is after review year (flagged): 24


,artist,album,releaseyear,reviewdate
124,The Avalanches,Since I Left You,2001,1999-12-31
532,Love Is All,Nine Times That Same Song,2006,2005-12-05
1215,Abilene,Abilene,2001,2000-12-31
3685,Cave In,Jupiter,2001,2000-08-08
4647,Alpha,The Impossible Thrill,2001,2000-12-31


### Repeat reviews of the same album

In [39]:
album_key = alb_data["artist"].str.casefold() + " || " + alb_data["album"].str.casefold()
alb_data = alb_data.assign(_album_key=album_key).sort_values(["_album_key", "reviewdate"])
alb_data["album_review_count"] = alb_data.groupby("_album_key")["_album_key"].transform("size")
alb_data["album_review_number"] = alb_data.groupby("_album_key").cumcount() + 1
alb_data = alb_data.drop(columns="_album_key").sort_values("source_row")

repeats = alb_data[alb_data["album_review_count"] > 1]
log("Repeat reviews", "Rows belonging to albums reviewed more than once (kept)", len(repeats))
display(repeats[["artist", "album", "reviewauthor", "score", "reviewdate", "is_reissue"]].head(8))

[Repeat reviews] Rows belonging to albums reviewed more than once (kept): 88


,artist,album,reviewauthor,score,reviewdate,is_reissue
12,American Football,American Football,Ian Cohen,8.6,2014-05-21,True
13,American Football,American Football,Ian Cohen,7.7,2016-10-26,False
99,Crystal Castles,Crystal Castles,Ian Cohen,7.8,2008-03-17,False
100,Crystal Castles,Crystal Castles,Ian Cohen,8.5,2010-04-29,False
321,Ty Segall,Ty Segall,Stuart Berman,8.0,2017-01-25,False
322,Ty Segall,Ty Segall,Stephen M. Deusner,6.7,2009-07-17,False
399,Atmosphere,God Loves Ugly,Nate Patrin,6.6,2009-02-20,True
400,Atmosphere,God Loves Ugly,Sam Chennault,5.7,2002-07-04,False


### Unique ID and column order

In [40]:
alb_data = alb_data.sort_values("source_row").reset_index(drop=True)
alb_data.insert(0, "review_id", np.arange(1, len(alb_data) + 1))

flag_cols = ["is_reissue", "release_after_review", "features_suspect",
             "loudness_positive", "tempo_unusual"]
helper_cols = ["years_since_release", "album_review_count", "album_review_number"]
core_cols = ["review_id", "source_row", "artist", "album", "reviewauthor", "score",
             "releaseyear", "reviewdate", "recordlabel", "genre"]
alb_data = alb_data[core_cols + FEATURE_COLS + helper_cols + flag_cols]
alb_data.head()

,review_id,source_row,artist,album,reviewauthor,score,releaseyear,reviewdate,recordlabel,genre,danceability,energy,loudness,speechiness,acousticness,instrumentalness,liveness,valence,tempo,years_since_release,album_review_count,album_review_number,is_reissue,release_after_review,features_suspect,loudness_positive,tempo_unusual
0,1,0,Studio 1,Studio 1,Andy Battaglia,8.5,2009,2009-02-18,Studio,Electronic,0.511917,0.499667,-5.626583,0.031983,0.724917,0.024493,0.165367,0.555083,101.395167,0,1,1,False,False,False,False,False
1,2,1,John Fahey,The Great Santa Barbara Oil Slick,Mark Richardson,8.2,2005,2005-02-13,Water,Folk/Country,0.369765,0.325412,-19.153824,0.148624,0.647053,0.559133,0.527782,0.179465,107.622647,0,1,1,False,False,False,False,False
2,3,2,Reigning Sound,Too Much Guitar,Stephen M. Deusner,8.3,2004,2004-08-19,In the Red,Electronic,0.253943,0.912857,-1.089500,0.055500,0.000253,0.751214,0.199071,0.552786,133.895500,0,1,1,False,False,False,False,False
3,4,3,The Red Thread,After the Last,Chris Dahlen,7.3,2003,2003-07-17,Badman,Rock,0.425400,0.433474,-12.871000,0.028260,0.310325,0.224137,0.125150,0.451400,104.354200,0,1,1,False,False,False,False,False
4,5,6,Jessica Lea Mayfield,With Blasphemy So Heartfelt,Stephen M. Deusner,8.2,2008,2008-11-11,Polymer,Rock,0.447583,0.399083,-8.402000,0.032267,0.694750,0.004033,0.123142,0.237933,111.549917,0,1,1,False,False,False,False,False


### Final Validation

In [41]:
assert alb_data["review_id"].is_unique, "review_id is not unique"
assert not alb_data.drop(columns=["review_id", "source_row"]).duplicated().any(), "Exact duplicates remain"
assert alb_data["score"].between(0, 10).all(), "Score outside 0-10"
assert alb_data["reviewdate"].notna().all(), "Unparsed review dates"
assert alb_data["releaseyear"].dropna().between(1900, alb_data["reviewdate"].dt.year.max() + 1).all(), "Implausible release year"
assert (alb_data[FEATURE_COLS] != -1).all().all(), "Sentinel -1 values remain"
assert ((alb_data[UNIT_INTERVAL_COLS] >= 0) & (alb_data[UNIT_INTERVAL_COLS] <= 1) | alb_data[UNIT_INTERVAL_COLS].isna()).all().all(), "Features outside [0, 1]"
for col in ["artist", "album", "reviewauthor", "recordlabel", "genre"]:
    s = alb_data[col].dropna()
    assert (s == s.str.strip()).all(), f"{col}: leading/trailing whitespace remains"
    assert not s.str.contains("  ").any(), f"{col}: double spaces remain"
    assert not s.str.lower().isin({p.lower() for p in MISSING_PLACEHOLDERS}).any(), f"{col}: placeholder strings remain"
if DROP_MISSING_KEY_FIELDS:
    assert alb_data[["artist", "album"]].notna().all().all(), "Missing artist/album remain"
print("All validation checks passed.")

All validation checks passed.


In [42]:
pd.DataFrame({
    "dtype": alb_data.dtypes.astype(str),
    "missing": alb_data.isna().sum(),
    "missing_%": (alb_data.isna().mean() * 100).round(2),
})

,dtype,missing,missing_%
review_id,int64,0,0.00
source_row,int64,0,0.00
artist,object,0,0.00
album,object,0,0.00
reviewauthor,object,0,0.00
score,float64,0,0.00
releaseyear,Int64,301,1.79
reviewdate,datetime64[ns],0,0.00
recordlabel,object,575,3.42
genre,object,1192,7.09


### Cleaning log and export

In [43]:
log_df = pd.DataFrame(cleaning_log)
log_df.to_csv(CLEANING_OUTPUT_DIR / "log_df.csv", index=False, encoding="utf-8")
print(f"Rows: {ROWS_AT_START:,} -> {len(alb_data):,}  ({ROWS_AT_START - len(alb_data):,} removed)")
log_df

Rows: 18,403 -> 16,811  (1,592 removed)


,step,detail,affected
0,Duplicates,Exact duplicate rows removed,2
1,Text normalisation,"artist: values changed (9,049 -> 9,047 unique)",140
2,Text normalisation,"album: values changed (17,714 -> 17,711 unique)",150
3,Text normalisation,reviewauthor: values changed (564 -> 559 unique),83
4,Text normalisation,"recordlabel: values changed (3,286 -> 3,283 un...",41
5,Text normalisation,genre: values changed (10 -> 10 unique),0
6,Disguised missing,artist: placeholder strings -> NaN,0
7,Disguised missing,album: placeholder strings -> NaN,0
8,Disguised missing,reviewauthor: placeholder strings -> NaN,0
9,Disguised missing,recordlabel: placeholder strings -> NaN,614


In [44]:
alb_data.to_csv(OUTPUT_PATH, index=False, encoding="utf-8", date_format="%Y-%m-%d")

check = pd.read_csv(OUTPUT_PATH, parse_dates=["reviewdate"])
assert len(check) == len(alb_data), "Row count changed on round-trip"
assert check["review_id"].is_unique
print(f"Saved {len(check):,} rows x {check.shape[1]} columns to {OUTPUT_PATH}")

Saved 16,811 rows x 27 columns to /home/ai4covid/test_stuff_2/outputs/data_clean.csv
